# **Qu'est ce que la base de données GDELT ?**
Le GDELT (Global Database of Events, Language, and Tone) est une initiative de recherche et une base de données gratuite en temps réel qui cartographie la société humaine mondiale. Il analyse quotidiennement des millions d'articles de presse, d'émissions et de documents du monde entier pour identifier les événements, les acteurs, le ton et les thèmes, permettant de visualiser les conflits, la politique et les tendances mondiales depuis 1979.

Notre objectifs premier ici est de pouvoir utiliser les données concernant le Bénin. Pour ce faire nous utiliserons Google BigQuery.

### **Rapidement, c'est quoi Google BigQuery ?**
C'est un entrepôt de données( datawarehouse ) utilisé pour faire des requêtes sur des volumes massifs de données.



In [1]:
# Installation des bibliothèques nécessaires pour Google Cloud BigQuery, pandas et pyarrow.
!pip install pandas pyarrow google-cloud-bigquery

Dans le processus d'extraction de données les ressources principalement utilisées sont:

*   https://docs.cloud.google.com/python/docs/reference/bigquery/latest
*   https://www.gdeltproject.org/data.html#documentation
*   https://blog.gdeltproject.org/
*   https://blog.gdeltproject.org/google-bigquery-gkg-2-0-sample-queries/


Pour notre projet nous allons principalement utilisés trois des bases de données de la GDELT :

*   GDELT 2.0 Event Database
*   GDELT 2.0 Global Knowledge Graph (GKG)
*   Visual Global Knowledge Graph (VGKG)

**La table Events de GDELT** recense les événements géopolitiques structurés détectés dans les médias du monde entier. Chaque ligne correspond à une interaction entre deux acteurs (pays, organisations, individus), avec des informations clés comme la date, le type d’action, la localisation, ainsi qu’un score de gravité (GoldsteinScale) et un ton moyen (AvgTone). Elle permet ainsi de capturer la réalité “factuelle” des événements.

À l’inverse, **la table GKG (Global Knowledge Graph)** analyse le contenu des articles de presse de manière plus riche et qualitative. Elle extrait des éléments comme les thèmes abordés, les personnes et organisations mentionnées, les lieux, ainsi que le ton médiatique détaillé (V2Tone). GKG permet donc de comprendre la perception et le narratif médiatique autour des événements.

Ainsi, les tables Events et GKG sont complémentaires : la première décrit les faits structurés, tandis que la seconde apporte une vision sémantique et contextuelle des informations diffusées dans les médias.

In [2]:
# Importation des bibliothèques Python requises.
import os
import pandas as pd
from datetime import datetime
from google.colab import auth
from google.colab import drive
from google.cloud import bigquery

# Authentification de l'utilisateur pour accéder aux services Google Cloud.
auth.authenticate_user()

In [3]:
# Monte Google Drive pour accéder aux fichiers stockés.

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **Explication des colonnes GDELT 2.0 Events**


### **IDENTIFIANTS / TEMPS**

* **GLOBALEVENTID** → ID unique de l’événement
* **SQLDATE** → date de l’événement (YYYYMMDD)
* **MonthYear** → mois + année (YYYYMM)
* **Year** → année seule
* **FractionDate** → date en format décimal (analyse fine du temps)
* **DATEADDED** → date d’ajout dans GDELT (pas la date réelle)


### **ACTEURS**

* **Actor1Code** → code de l’acteur principal
* **Actor1Name** → nom acteur principal
* **Actor1CountryCode** → pays acteur 1
* **Actor1KnownGroupCode** → groupe connu (ONG, armée…)
* **Actor1EthnicCode** → groupe ethnique acteur 1
* **Actor1Religion1Code / 2** → religion acteur 1
* **Actor1Type1/2/3Code** → type (gouvernement, militaire, civil…)
* **Actor2Code** → code acteur secondaire
* **Actor2Name** → nom acteur secondaire
* **Actor2CountryCode** → pays acteur 2
* **Actor2KnownGroupCode / Ethnic / Religion / Type** → mêmes infos pour acteur


### **TYPE D’ÉVÉNEMENT**

* **IsRootEvent** → événement principal ou dérivé
* **EventCode** → code précis de l’action
* **EventBaseCode** → version simplifiée
* **EventRootCode** → catégorie générale
* **QuadClass** → type global (coopération / conflit verbal / matériel)


### **IMPACT / IMPORTANCE**

* **GoldsteinScale** → gravité de -10 (conflit) à +10 (coopération)
* **NumMentions** → nombre de mentions dans les médias
* **NumSources** → nombre de sources différentes
* **NumArticles** → nombre d’articles
* **AvgTone** → sentiment global (négatif = crise, positif = stable)


### **LOCALISATION ACTEUR 1**

* **Actor1Geo_Type** → type de lieu (ville, pays…)
* **Actor1Geo_FullName** → nom du lieu
* **Actor1Geo_CountryCode** → pays
* **Actor1Geo_ADM1Code** → région/état
* **Actor1Geo_ADM2Code** → sous-région
* **Actor1Geo_Lat / Long** → coordonnées
* **Actor1Geo_FeatureID** → ID géographique


### **LOCALISATION DE L’ÉVÉNEMENT (IMPORTANT)**

* **ActionGeo_Type** → niveau du lieu réel
* **ActionGeo_FullName** → nom complet du lieu
* **ActionGeo_CountryCode** → pays (clé pour ton projet Bénin)
* **ActionGeo_ADM1Code / ADM2Code** → région/sous-région
* **ActionGeo_Lat / Long** → coordonnées GPS
* **ActionGeo_FeatureID** → ID géographique


### **SOURCE**

* **SOURCEURL** → lien vers l’article original (très important pour RAG + vérification info)


In [7]:
# Importation de la base de données GDELT 2.0 Event depuis un fichier CSV.
# Veuillez vous assurer que le chemin du fichier est correct et que le fichier existe.
events_2025 = pd.read_csv('/content/drive/MyDrive/bq-results-20260429-094608-1777455985672/events_2025.csv')

# Affichage des premières lignes pour un aperçu des données.
events_2025.head()

,GLOBALEVENTID,SQLDATE,MonthYear,Year,FractionDate,Actor1Code,Actor1Name,Actor1CountryCode,Actor1KnownGroupCode,Actor1EthnicCode,...,ActionGeo_Type,ActionGeo_FullName,ActionGeo_CountryCode,ActionGeo_ADM1Code,ActionGeo_ADM2Code,ActionGeo_Lat,ActionGeo_Long,ActionGeo_FeatureID,DATEADDED,SOURCEURL
0,1289290869,20250215,202502,2025,2025.1233,AFRCVL,AFRICA,AFR,NaN,NaN,...,1,Benin,BN,BN,NaN,9.50,2.25,BN,20260215143000,https://www.newsghana.com.gh/african-leaders-c...
1,1292975459,20250307,202503,2025,2025.1836,NaN,NaN,NaN,NaN,NaN,...,5,"Borgou, Borgou, Benin",BN,BN10,5886.0,9.75,2.75,-1332814,20260307140000,http://www.nigeriasun.com/news/278907556/benin...
2,1292317038,20250304,202503,2025,2025.1753,OPP,PRISONER,NaN,NaN,NaN,...,1,Benin,BN,BN,NaN,9.50,2.25,BN,20260304111500,https://lanouvelletribune.info/2026/03/benin-d...
3,1290366412,20250221,202502,2025,2025.1397,UAF,GUNMEN,NaN,NaN,NaN,...,1,Benin,BN,BN,NaN,9.50,2.25,BN,20260221200000,https://www.premiumtimesng.com/regional/south-...
4,1297200929,20250401,202504,2025,2025.2493,USA,UNITED STATES,USA,NaN,NaN,...,1,Benin,BN,BN,NaN,9.50,2.25,BN,20260401164500,https://www.mediapool.bg/vissh-amerikanski-slu...


# **Sélection des caractéristiques pour les événements GDELT**

In [8]:
# Affichage des noms de colonnes et des dimensions (nombre de lignes, nombre de colonnes) du DataFrame 'events_2025'.
print(events_2025.columns)
print(events_2025.shape)

Index(['GLOBALEVENTID', 'SQLDATE', 'MonthYear', 'Year', 'FractionDate',
       'Actor1Code', 'Actor1Name', 'Actor1CountryCode', 'Actor1KnownGroupCode',
       'Actor1EthnicCode', 'Actor1Religion1Code', 'Actor1Religion2Code',
       'Actor1Type1Code', 'Actor1Type2Code', 'Actor1Type3Code', 'Actor2Code',
       'Actor2Name', 'Actor2CountryCode', 'Actor2KnownGroupCode',
       'Actor2EthnicCode', 'Actor2Religion1Code', 'Actor2Religion2Code',
       'Actor2Type1Code', 'Actor2Type2Code', 'Actor2Type3Code', 'IsRootEvent',
       'EventCode', 'EventBaseCode', 'EventRootCode', 'QuadClass',
       'GoldsteinScale', 'NumMentions', 'NumSources', 'NumArticles', 'AvgTone',
       'Actor1Geo_Type', 'Actor1Geo_FullName', 'Actor1Geo_CountryCode',
       'Actor1Geo_ADM1Code', 'Actor1Geo_ADM2Code', 'Actor1Geo_Lat',
       'Actor1Geo_Long', 'Actor1Geo_FeatureID', 'Actor2Geo_Type',
       'Actor2Geo_FullName', 'Actor2Geo_CountryCode', 'Actor2Geo_ADM1Code',
       'Actor2Geo_ADM2Code', 'Actor2Geo_Lat', 'Act

**Nous allons conserver les colonnes suivantes de la base de données GDELT 2.0 Events:**
*   `SQLDATE`: Date de l'événement.
*   `Actor1Name`: Nom de l'acteur principal.
*   `Actor2Name`: Nom de l'acteur secondaire.
*   `Actor1CountryCode`: Code pays de l'acteur principal.
*   `Actor2CountryCode`: Code pays de l'acteur secondaire.
*   `IsRootEvent`: Indique si l'événement est principal ou dérivé.
*   `EventCode`: Code précis de l'action.
*   `EventRootCode`: Catégorie générale de l'action.
*   `GoldsteinScale`: Échelle de Goldstein (gravité de -10 à +10).
*   `AvgTone`: Tonalité moyenne de l'événement.
*   `NumMentions`: Nombre de mentions dans les médias.
*   `NumSources`: Nombre de sources différentes.
*   `NumArticles`: Nombre d'articles.
*   `ActionGeo_CountryCode`: Code pays du lieu de l'action.
*   `ActionGeo_FullName`: Nom complet du lieu de l'action.
*   `SOURCEURL`: URL de la source originale.

In [11]:
# Définition de la liste des colonnes à conserver du DataFrame events_2025.
columns_to_keep = ['SQLDATE', 'Actor1Name', 'Actor2Name',
                   'Actor1CountryCode', 'Actor2CountryCode', 'IsRootEvent',
                   'EventCode', 'EventRootCode', 'GoldsteinScale', 'AvgTone', 'NumMentions',
                   'NumSources', 'NumArticles', 'ActionGeo_CountryCode',
                   'ActionGeo_FullName', 'SOURCEURL']

In [12]:
# Création d'un nouveau DataFrame 'events' avec uniquement les colonnes sélectionnées.
events = events_2025[columns_to_keep]
# Affichage des premières lignes du nouveau DataFrame.
events.head()

,SQLDATE,Actor1Name,Actor2Name,Actor1CountryCode,Actor2CountryCode,IsRootEvent,EventCode,EventRootCode,GoldsteinScale,AvgTone,NumMentions,NumSources,NumArticles,ActionGeo_CountryCode,ActionGeo_FullName,SOURCEURL
0,20250215,AFRICA,GHANA,AFR,GHA,0,841,8,7.0,1.218027,4,1,4,BN,Benin,https://www.newsghana.com.gh/african-leaders-c...
1,20250307,NaN,NIGERIAN,NaN,NGA,1,190,19,-10.0,-5.817610,10,1,10,BN,"Borgou, Borgou, Benin",http://www.nigeriasun.com/news/278907556/benin...
2,20250304,PRISONER,NaN,NaN,NaN,1,22,2,3.2,-2.621723,10,1,10,BN,Benin,https://lanouvelletribune.info/2026/03/benin-d...
3,20250221,GUNMEN,NaN,NaN,NaN,0,181,18,-9.0,-8.928571,10,1,10,BN,Benin,https://www.premiumtimesng.com/regional/south-...
4,20250401,UNITED STATES,NaN,USA,NaN,1,10,1,0.0,-5.555556,4,1,4,BN,Benin,https://www.mediapool.bg/vissh-amerikanski-slu...


In [13]:
# Importation de la base de données GDELT 2.0 Global Knowledge Graph (GKG) depuis un fichier CSV.
gkg = pd.read_csv('/content/drive/MyDrive/bq-results-20260429-095826-1777456740233/gkg_2025.csv')
# Affichage des premières lignes pour un aperçu des données.
gkg.head()

,DATE,DocumentIdentifier,V2Themes,V2Tone,V2Locations,V2Persons,V2Organizations,V2Counts,SharingImage,RelatedImages
0,20251202161500,https://fr.allafrica.com/stories/202512020485....,"SLFID_NATURAL_RESOURCES,1013;WB_471_ECONOMIC_G...","5.94405594405594,5.94405594405594,0,5.94405594...",1#Niger#NG#NG##16#8#NG#44;1#Niger#NG#NG##16#8#...,NaN,"Authority Of Basin Of Niger Them,49;Authority ...",NaN,NaN,NaN
1,20251202161500,https://fr.allafrica.com/stories/202512020413....,NaN,"1.25944584382872,3.27455919395466,2.0151133501...",1#Seychelles#SE#SE##-4.583333#55.666667#SE#167...,NaN,"Federation Madagascar,1180",NaN,NaN,NaN
2,20250702100000,https://malijet.com/actualite-sur-afrique/3034...,"NATURAL_DISASTER_DROWNING,1574;SLFID_ECONOMIC_...","-3.25,2.75,6,8.75,17.5,0,377",1#Benin#BN#BN##9.5#2.25#BN#885;1#Benin#BN#BN##...,NaN,"A Committee,112",KILL#2#citizens Benin#1#Togo#TO#TO#8#1.166667#...,NaN,NaN
3,20251019013000,https://fightnews.com/2025-pfl-africa-semifina...,"TAX_FNCACT_REFEREE,2609;TAX_FNCACT_REFEREE,452...","-0.4739336492891,3.69668246445498,4.1706161137...",1#Cameroon#CM#CM##6#12#CM#1901;1#Cameroon#CM#C...,"Africa Smartcage,965;Africa Justin Clarke,2800...","Fighters League,5388",PROTEST#2014##1#Cameroon#CM#CM#6#12#CM#2837;TA...,NaN,NaN
4,20251019001500,https://theeagleonline.com.ng/cultural-capital...,"EPU_ECONOMY,3599;EPU_ECONOMY,5460;EPU_ECONOMY_...","5.84629891560585,6.69495520980669,0.8486562942...","1#Nigerians#NI#NI##10#8#NI#9474;4#Lagos, Lagos...","Bilal Akkouche,9797;Ladi Kwali,10469;Aina Onab...","Coronation Group,97;Coronation Group,1535;Coro...",NaN,https://theeagleonline.com.ng/wp-content/uploa...,NaN


In [14]:
# Affichage des noms de colonnes et des dimensions du DataFrame 'gkg'.
print(gkg.columns)
print(gkg.shape)

Index(['DATE', 'DocumentIdentifier', 'V2Themes', 'V2Tone', 'V2Locations',
       'V2Persons', 'V2Organizations', 'V2Counts', 'SharingImage',
       'RelatedImages'],
      dtype='object')
(48041, 10)


In [15]:
# Affichage d'informations concises sur le DataFrame 'events', y compris les types de données et les valeurs non nulles.
events.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23859 entries, 0 to 23858
Data columns (total 16 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   SQLDATE                23859 non-null  int64  
 1   Actor1Name             21599 non-null  object 
 2   Actor2Name             16627 non-null  object 
 3   Actor1CountryCode      12226 non-null  object 
 4   Actor2CountryCode      10045 non-null  object 
 5   IsRootEvent            23859 non-null  int64  
 6   EventCode              23859 non-null  int64  
 7   EventRootCode          23859 non-null  int64  
 8   GoldsteinScale         23859 non-null  float64
 9   AvgTone                23859 non-null  float64
 10  NumMentions            23859 non-null  int64  
 11  NumSources             23859 non-null  int64  
 12  NumArticles            23859 non-null  int64  
 13  ActionGeo_CountryCode  23859 non-null  object 
 14  ActionGeo_FullName     23859 non-null  object 
 15  SO

# **NETTOYAGE DES DATASETS**

Cette section est dédiée au nettoyage et à la préparation des données des DataFrames `events` et `gkg` pour les rendre utilisables pour l'analyse. Cela inclut la conversion des types de données, la gestion des valeurs manquantes et la standardisation des formats.

In [16]:
# Conversion de la colonne 'SQLDATE' en format de date (YYYY-MM-DD).
events["SQLDATE"] = pd.to_datetime(events["SQLDATE"], format="%Y%m%d")

/tmp/ipykernel_11193/3109013807.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  events["SQLDATE"] = pd.to_datetime(events["SQLDATE"], format="%Y%m%d")


In [17]:
# Suppression des lignes où 'Actor1Name', 'EventCode' ou 'GoldsteinScale' sont manquants, car ce sont des informations essentielles.
events = events.dropna(subset=["Actor1Name", "EventCode", "GoldsteinScale"])

In [18]:
# Remplacement des valeurs manquantes dans 'Actor2Name' par 'UNKNOWN' pour maintenir la cohérence.
events["Actor2Name"] = events["Actor2Name"].fillna("UNKNOWN")

In [19]:
# Affichage du DataFrame 'events' après les opérations de nettoyage.
events

,SQLDATE,Actor1Name,Actor2Name,Actor1CountryCode,Actor2CountryCode,IsRootEvent,EventCode,EventRootCode,GoldsteinScale,AvgTone,NumMentions,NumSources,NumArticles,ActionGeo_CountryCode,ActionGeo_FullName,SOURCEURL
0,2025-02-15,AFRICA,GHANA,AFR,GHA,0,841,8,7.0,1.218027,4,1,4,BN,Benin,https://www.newsghana.com.gh/african-leaders-c...
2,2025-03-04,PRISONER,UNKNOWN,NaN,NaN,1,22,2,3.2,-2.621723,10,1,10,BN,Benin,https://lanouvelletribune.info/2026/03/benin-d...
3,2025-02-21,GUNMEN,UNKNOWN,NaN,NaN,0,181,18,-9.0,-8.928571,10,1,10,BN,Benin,https://www.premiumtimesng.com/regional/south-...
4,2025-04-01,UNITED STATES,UNKNOWN,USA,NaN,1,10,1,0.0,-5.555556,4,1,4,BN,Benin,https://www.mediapool.bg/vissh-amerikanski-slu...
5,2025-04-13,INTERNATIONAL MONETARY FUND,UNKNOWN,NaN,NaN,0,51,5,3.4,-6.821963,5,1,5,BN,Benin,https://allafrica.com/stories/202604130011.html
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23854,2025-09-15,AFRICA,BENIN,AFR,BEN,1,42,4,1.9,5.952381,2,1,2,BN,Benin,https://lequotidien.sn/musique-2e-edition-du-s...
23855,2025-09-15,UNITED STATES,GHANA,USA,GHA,0,60,6,6.0,-3.726469,1,1,1,BN,Benin,https://www.eurasiareview.com/15092025-arms-ra...
23856,2025-09-15,POLICE,NIGERIA,NaN,NGA,1,10,1,0.0,-7.296137,10,1,10,BN,Benin,https://promptnewsonline.com/police-detain-off...
23857,2025-09-15,POLICE COMMISSIONER,FURNITURE MAKER,NaN,NaN,1,173,17,-5.0,-6.578947,3,1,3,BN,Benin,https://thenationonlineng.net/police-detain-of...


In [ ]:
# Conversion de 'Actor1Name' en minuscules et suppression des espaces blancs pour la standardisation.
events["Actor1Name"] = events["Actor1Name"].str.lower().str.strip()

In [21]:
# Suppression des lignes où 'Actor1CountryCode' est manquant, car cette colonne est cruciale pour le filtrage par pays.
events = events.dropna(subset=["Actor1CountryCode"])

In [22]:
# Remplacement des valeurs manquantes dans 'Actor2CountryCode' par 'UNKNOWN' pour maintenir la cohérence.
events["Actor2CountryCode"] = events["Actor2CountryCode"].fillna("UNKNOWN")

/tmp/ipykernel_11193/1124160468.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  events["Actor2CountryCode"] = events["Actor2CountryCode"].fillna("UNKNOWN")


In [23]:
# Affichage du DataFrame 'events' après les opérations de nettoyage.
events

,SQLDATE,Actor1Name,Actor2Name,Actor1CountryCode,Actor2CountryCode,IsRootEvent,EventCode,EventRootCode,GoldsteinScale,AvgTone,NumMentions,NumSources,NumArticles,ActionGeo_CountryCode,ActionGeo_FullName,SOURCEURL
0,2025-02-15,africa,GHANA,AFR,GHA,0,841,8,7.0,1.218027,4,1,4,BN,Benin,https://www.newsghana.com.gh/african-leaders-c...
4,2025-04-01,united states,UNKNOWN,USA,UNKNOWN,1,10,1,0.0,-5.555556,4,1,4,BN,Benin,https://www.mediapool.bg/vissh-amerikanski-slu...
6,2025-04-16,benin,REBELLION,BEN,UNKNOWN,1,100,10,-5.0,-12.500000,6,1,6,BN,Benin,https://article.wn.com/view/2026/04/16/Benin_a...
7,2025-04-16,france,MILITARY,FRA,UNKNOWN,0,51,5,3.4,-6.829268,2,1,2,BN,Benin,https://www.taipeitimes.com/News/world/archive...
8,2025-04-16,nigeria,MILITARY,NGA,UNKNOWN,0,51,5,3.4,-6.829268,2,1,2,BN,Benin,https://www.taipeitimes.com/News/world/archive...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23852,2025-09-15,benin,AFRICA,BEN,AFR,1,42,4,1.9,6.770833,6,1,6,BN,Benin,https://fr.allafrica.com/stories/202509150347....
23853,2025-09-15,benin,AFRICA,BEN,AFR,1,42,4,1.9,6.770833,2,1,2,BN,Benin,https://fr.allafrica.com/stories/202509150347....
23854,2025-09-15,africa,BENIN,AFR,BEN,1,42,4,1.9,5.952381,2,1,2,BN,Benin,https://lequotidien.sn/musique-2e-edition-du-s...
23855,2025-09-15,united states,GHANA,USA,GHA,0,60,6,6.0,-3.726469,1,1,1,BN,Benin,https://www.eurasiareview.com/15092025-arms-ra...


In [24]:
# Conversion de 'Actor2Name' en minuscules et suppression des espaces blancs pour la standardisation.
events["Actor2Name"] = events["Actor2Name"].str.lower().str.strip()

/tmp/ipykernel_11193/3485392073.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  events["Actor2Name"] = events["Actor2Name"].str.lower().str.strip()


In [25]:
# Affichage du DataFrame 'events' après les opérations de nettoyage.
events

,SQLDATE,Actor1Name,Actor2Name,Actor1CountryCode,Actor2CountryCode,IsRootEvent,EventCode,EventRootCode,GoldsteinScale,AvgTone,NumMentions,NumSources,NumArticles,ActionGeo_CountryCode,ActionGeo_FullName,SOURCEURL
0,2025-02-15,africa,ghana,AFR,GHA,0,841,8,7.0,1.218027,4,1,4,BN,Benin,https://www.newsghana.com.gh/african-leaders-c...
4,2025-04-01,united states,unknown,USA,UNKNOWN,1,10,1,0.0,-5.555556,4,1,4,BN,Benin,https://www.mediapool.bg/vissh-amerikanski-slu...
6,2025-04-16,benin,rebellion,BEN,UNKNOWN,1,100,10,-5.0,-12.500000,6,1,6,BN,Benin,https://article.wn.com/view/2026/04/16/Benin_a...
7,2025-04-16,france,military,FRA,UNKNOWN,0,51,5,3.4,-6.829268,2,1,2,BN,Benin,https://www.taipeitimes.com/News/world/archive...
8,2025-04-16,nigeria,military,NGA,UNKNOWN,0,51,5,3.4,-6.829268,2,1,2,BN,Benin,https://www.taipeitimes.com/News/world/archive...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23852,2025-09-15,benin,africa,BEN,AFR,1,42,4,1.9,6.770833,6,1,6,BN,Benin,https://fr.allafrica.com/stories/202509150347....
23853,2025-09-15,benin,africa,BEN,AFR,1,42,4,1.9,6.770833,2,1,2,BN,Benin,https://fr.allafrica.com/stories/202509150347....
23854,2025-09-15,africa,benin,AFR,BEN,1,42,4,1.9,5.952381,2,1,2,BN,Benin,https://lequotidien.sn/musique-2e-edition-du-s...
23855,2025-09-15,united states,ghana,USA,GHA,0,60,6,6.0,-3.726469,1,1,1,BN,Benin,https://www.eurasiareview.com/15092025-arms-ra...


In [26]:
# Conversion de 'Actor1CountryCode' et 'Actor2CountryCode' en minuscules et suppression des espaces blancs.
events["Actor1CountryCode"] = events["Actor1CountryCode"].str.lower().str.strip()
events["Actor2CountryCode"] = events["Actor2CountryCode"].str.lower().str.strip()

/tmp/ipykernel_11193/1516816878.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  events["Actor1CountryCode"] = events["Actor1CountryCode"].str.lower().str.strip()
/tmp/ipykernel_11193/1516816878.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  events["Actor2CountryCode"] = events["Actor2CountryCode"].str.lower().str.strip()


In [27]:
# Affichage du DataFrame 'events' après les opérations de nettoyage.
events

,SQLDATE,Actor1Name,Actor2Name,Actor1CountryCode,Actor2CountryCode,IsRootEvent,EventCode,EventRootCode,GoldsteinScale,AvgTone,NumMentions,NumSources,NumArticles,ActionGeo_CountryCode,ActionGeo_FullName,SOURCEURL
0,2025-02-15,africa,ghana,afr,gha,0,841,8,7.0,1.218027,4,1,4,BN,Benin,https://www.newsghana.com.gh/african-leaders-c...
4,2025-04-01,united states,unknown,usa,unknown,1,10,1,0.0,-5.555556,4,1,4,BN,Benin,https://www.mediapool.bg/vissh-amerikanski-slu...
6,2025-04-16,benin,rebellion,ben,unknown,1,100,10,-5.0,-12.500000,6,1,6,BN,Benin,https://article.wn.com/view/2026/04/16/Benin_a...
7,2025-04-16,france,military,fra,unknown,0,51,5,3.4,-6.829268,2,1,2,BN,Benin,https://www.taipeitimes.com/News/world/archive...
8,2025-04-16,nigeria,military,nga,unknown,0,51,5,3.4,-6.829268,2,1,2,BN,Benin,https://www.taipeitimes.com/News/world/archive...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23852,2025-09-15,benin,africa,ben,afr,1,42,4,1.9,6.770833,6,1,6,BN,Benin,https://fr.allafrica.com/stories/202509150347....
23853,2025-09-15,benin,africa,ben,afr,1,42,4,1.9,6.770833,2,1,2,BN,Benin,https://fr.allafrica.com/stories/202509150347....
23854,2025-09-15,africa,benin,afr,ben,1,42,4,1.9,5.952381,2,1,2,BN,Benin,https://lequotidien.sn/musique-2e-edition-du-s...
23855,2025-09-15,united states,ghana,usa,gha,0,60,6,6.0,-3.726469,1,1,1,BN,Benin,https://www.eurasiareview.com/15092025-arms-ra...


In [28]:
# Filtrage du DataFrame 'events' pour ne conserver que les événements où l'acteur principal est le Bénin ('ben').
events = events[events["Actor1CountryCode"] == "ben"]

In [29]:
# Affichage du DataFrame 'events' filtré.
events

,SQLDATE,Actor1Name,Actor2Name,Actor1CountryCode,Actor2CountryCode,IsRootEvent,EventCode,EventRootCode,GoldsteinScale,AvgTone,NumMentions,NumSources,NumArticles,ActionGeo_CountryCode,ActionGeo_FullName,SOURCEURL
6,2025-04-16,benin,rebellion,ben,unknown,1,100,10,-5.0,-12.500000,6,1,6,BN,Benin,https://article.wn.com/view/2026/04/16/Benin_a...
9,2025-04-16,benin,russia,ben,rus,1,100,10,-5.0,-12.500000,4,1,4,BN,Benin,https://article.wn.com/view/2026/04/16/Benin_a...
15,2025-03-27,benin,radio station,ben,unknown,0,190,19,-10.0,-0.946372,3,1,3,BN,"Banikoara, Alibori, Benin",https://www.swissinfo.ch/eng/war-peace/women-h...
27,2025-08-24,benin,unknown,ben,unknown,1,17,1,0.0,-2.314815,15,1,10,BN,"Ouidah, Atlantique, Benin",https://english.news.cn/africa/20250824/f847dc...
28,2025-08-24,beninese,unknown,ben,unknown,1,174,17,-5.0,-2.314815,7,1,7,BN,"Ouidah, Atlantique, Benin",https://english.news.cn/africa/20250824/f847dc...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23847,2025-09-15,benin,actor,ben,unknown,1,20,2,3.0,5.952381,5,1,5,BN,Benin,https://lequotidien.sn/musique-2e-edition-du-s...
23848,2025-09-15,benin,government,ben,unknown,0,150,15,-7.2,0.000000,4,1,4,BN,Benin,https://nigerianobservernews.com/2025/09/edo-g...
23851,2025-09-15,benin,benin,ben,ben,0,20,2,3.0,-1.417526,10,1,10,BN,"Malanville, Atakora, Benin",https://lanouvelletribune.info/2025/09/aspiran...
23852,2025-09-15,benin,africa,ben,afr,1,42,4,1.9,6.770833,6,1,6,BN,Benin,https://fr.allafrica.com/stories/202509150347....


# **NETTOYAGE DU DATASET GKG**

Cette section est dédiée au nettoyage et à la préparation des données du DataFrame `gkg`.

In [30]:
# Affichage des premières lignes du DataFrame 'gkg' pour un aperçu.
gkg.head()

,DATE,DocumentIdentifier,V2Themes,V2Tone,V2Locations,V2Persons,V2Organizations,V2Counts,SharingImage,RelatedImages
0,20251202161500,https://fr.allafrica.com/stories/202512020485....,"SLFID_NATURAL_RESOURCES,1013;WB_471_ECONOMIC_G...","5.94405594405594,5.94405594405594,0,5.94405594...",1#Niger#NG#NG##16#8#NG#44;1#Niger#NG#NG##16#8#...,NaN,"Authority Of Basin Of Niger Them,49;Authority ...",NaN,NaN,NaN
1,20251202161500,https://fr.allafrica.com/stories/202512020413....,NaN,"1.25944584382872,3.27455919395466,2.0151133501...",1#Seychelles#SE#SE##-4.583333#55.666667#SE#167...,NaN,"Federation Madagascar,1180",NaN,NaN,NaN
2,20250702100000,https://malijet.com/actualite-sur-afrique/3034...,"NATURAL_DISASTER_DROWNING,1574;SLFID_ECONOMIC_...","-3.25,2.75,6,8.75,17.5,0,377",1#Benin#BN#BN##9.5#2.25#BN#885;1#Benin#BN#BN##...,NaN,"A Committee,112",KILL#2#citizens Benin#1#Togo#TO#TO#8#1.166667#...,NaN,NaN
3,20251019013000,https://fightnews.com/2025-pfl-africa-semifina...,"TAX_FNCACT_REFEREE,2609;TAX_FNCACT_REFEREE,452...","-0.4739336492891,3.69668246445498,4.1706161137...",1#Cameroon#CM#CM##6#12#CM#1901;1#Cameroon#CM#C...,"Africa Smartcage,965;Africa Justin Clarke,2800...","Fighters League,5388",PROTEST#2014##1#Cameroon#CM#CM#6#12#CM#2837;TA...,NaN,NaN
4,20251019001500,https://theeagleonline.com.ng/cultural-capital...,"EPU_ECONOMY,3599;EPU_ECONOMY,5460;EPU_ECONOMY_...","5.84629891560585,6.69495520980669,0.8486562942...","1#Nigerians#NI#NI##10#8#NI#9474;4#Lagos, Lagos...","Bilal Akkouche,9797;Ladi Kwali,10469;Aina Onab...","Coronation Group,97;Coronation Group,1535;Coro...",NaN,https://theeagleonline.com.ng/wp-content/uploa...,NaN


In [31]:
# Filtrage du DataFrame 'gkg' pour ne conserver que les entrées mentionnant 'Benin' dans 'V2Locations'.
gkg = gkg[gkg["V2Locations"].str.contains("Benin", na=False)]

In [32]:
# Conversion de la colonne 'DATE' en format de date, en gérant les erreurs de conversion.
gkg["DATE"] = pd.to_datetime(gkg["DATE"], errors="coerce")

In [33]:
# Remplacement des valeurs manquantes dans 'V2Tone' par 0.
gkg["V2Tone"] = gkg["V2Tone"].fillna(0)

In [34]:
# Extraction de la tonalité principale (premier élément) de la colonne 'V2Tone' et conversion en float.
gkg["tone"] = gkg["V2Tone"].astype(str).apply(lambda x: float(x.split(",")[0]))

In [35]:
# Extraction du score positif, du score négatif et de la polarité à partir de la colonne 'V2Tone'.
gkg["positive_score"] = gkg["V2Tone"].astype(str).apply(lambda x: float(x.split(",")[1]) if len(x.split(",")) > 1 else 0)
gkg["negative_score"] = gkg["V2Tone"].astype(str).apply(lambda x: float(x.split(",")[2]) if len(x.split(",")) > 2 else 0)
gkg["polarity"] = gkg["V2Tone"].astype(str).apply(lambda x: float(x.split(",")[3]) if len(x.split(",")) > 2 else 0)

In [36]:
# Suppression de la colonne originale 'V2Tone' car ses informations ont été extraites dans de nouvelles colonnes.
gkg = gkg.drop(columns=["V2Tone"])

In [37]:
# Filtrage du DataFrame 'gkg' pour ne conserver que les entrées mentionnant 'Benin' dans 'V2Locations'.
# Note: Cette opération a déjà été effectuée précédemment. La maintenir assure que seule les entrées 'Benin' sont présentes.
gkg = gkg[gkg["V2Locations"].str.contains("Benin", na=False)]

In [38]:
# Transformation de la colonne 'V2Themes' en listes de thèmes en utilisant le séparateur ';'.
gkg["themes"] = gkg["V2Themes"].astype(str).apply(lambda x: x.split(";"))

In [39]:
# Affichage du DataFrame 'gkg' après les opérations de nettoyage.
gkg

,DATE,DocumentIdentifier,V2Themes,V2Locations,V2Persons,V2Organizations,V2Counts,SharingImage,RelatedImages,tone,positive_score,negative_score,polarity,themes
0,1970-01-01 05:37:31.202161500,https://fr.allafrica.com/stories/202512020485....,"SLFID_NATURAL_RESOURCES,1013;WB_471_ECONOMIC_G...",1#Niger#NG#NG##16#8#NG#44;1#Niger#NG#NG##16#8#...,NaN,"Authority Of Basin Of Niger Them,49;Authority ...",NaN,NaN,NaN,5.944056,5.944056,0.000000,5.944056,"[SLFID_NATURAL_RESOURCES,1013, WB_471_ECONOMIC..."
1,1970-01-01 05:37:31.202161500,https://fr.allafrica.com/stories/202512020413....,NaN,1#Seychelles#SE#SE##-4.583333#55.666667#SE#167...,NaN,"Federation Madagascar,1180",NaN,NaN,NaN,1.259446,3.274559,2.015113,5.289673,[nan]
2,1970-01-01 05:37:30.702100000,https://malijet.com/actualite-sur-afrique/3034...,"NATURAL_DISASTER_DROWNING,1574;SLFID_ECONOMIC_...",1#Benin#BN#BN##9.5#2.25#BN#885;1#Benin#BN#BN##...,NaN,"A Committee,112",KILL#2#citizens Benin#1#Togo#TO#TO#8#1.166667#...,NaN,NaN,-3.250000,2.750000,6.000000,8.750000,"[NATURAL_DISASTER_DROWNING,1574, SLFID_ECONOMI..."
3,1970-01-01 05:37:31.019013000,https://fightnews.com/2025-pfl-africa-semifina...,"TAX_FNCACT_REFEREE,2609;TAX_FNCACT_REFEREE,452...",1#Cameroon#CM#CM##6#12#CM#1901;1#Cameroon#CM#C...,"Africa Smartcage,965;Africa Justin Clarke,2800...","Fighters League,5388",PROTEST#2014##1#Cameroon#CM#CM#6#12#CM#2837;TA...,NaN,NaN,-0.473934,3.696682,4.170616,7.867299,"[TAX_FNCACT_REFEREE,2609, TAX_FNCACT_REFEREE,4..."
4,1970-01-01 05:37:31.019001500,https://theeagleonline.com.ng/cultural-capital...,"EPU_ECONOMY,3599;EPU_ECONOMY,5460;EPU_ECONOMY_...","1#Nigerians#NI#NI##10#8#NI#9474;4#Lagos, Lagos...","Bilal Akkouche,9797;Ladi Kwali,10469;Aina Onab...","Coronation Group,97;Coronation Group,1535;Coro...",NaN,https://theeagleonline.com.ng/wp-content/uploa...,NaN,5.846299,6.694955,0.848656,7.543612,"[EPU_ECONOMY,3599, EPU_ECONOMY,5460, EPU_ECONO..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48036,1970-01-01 05:37:30.612160000,https://www.miragenews.com/iaea-fao-launch-ato...,"POVERTY,2111;WB_695_POVERTY,2111;WB_427_WATER_...","4#Loumbila, Region Du Plateau-Central, Burkina...","Rafael Mariano Grossi,2614;Dongxin Feng,3807","International Atomic Energy Agency,108;Centre ...",NaN,https://cdn1.miragenews.com/wp-content/uploads...,NaN,1.610738,4.697987,3.087248,7.785235,"[POVERTY,2111, WB_695_POVERTY,2111, WB_427_WAT..."
48037,1970-01-01 05:37:30.612160000,https://www.aa.com.tr/en/world/south-africa-re...,"SANCTIONS,68;SANCTIONS,921;SANCTIONS,1428;CRIS...",1#Israeli#IS#IS##31.5#34.75#IS#2049;1#Benin#BN...,"Antonio Guterres,1742;Benjamin Netanyahu,2083;...","African Foreign Ministry,332;International Cri...",NaN,https://cdnuploads.aa.com.tr/uploads/Contents/...,NaN,-10.000000,2.121212,12.121212,14.242424,"[SANCTIONS,68, SANCTIONS,921, SANCTIONS,1428, ..."
48038,1970-01-01 05:37:30.612134500,https://www.salon.com/2025/06/12/the-age-of-gl...,"SCIENCE,2325;TAX_FNCACT_SCIENTIST,2325;TAX_FNC...",1#Moldova#MD#MD##47#29#MD#12766;1#Somalia#SO#S...,"David Ben-Gurion,6873;Mahmoud Khalil,13522;Pat...","Peel Commission,6786;United Nations,11347;Unit...","KILL#20000##4#Gaza, Israel (General), Israel#I...",https://mediaproxy.salon.com/width/1200/https:...,NaN,-4.111153,2.740769,6.851922,9.592691,"[SCIENCE,2325, TAX_FNCACT_SCIENTIST,2325, TAX_..."
48039,1970-01-01 05:37:30.612140000,https://hunan.voc.com.cn/news/202506/29689492....,"TAX_FNCACT_SEAMAN,555;TAX_FNCACT_VENDOR,2745;T...","4#Xiashou, Guangdong, China#CH#CH30#13057#22.3...","Zebian King,6593;Hill Sea,336","Park Joe,4813;Benin Confucius Institute,1615;J...",NaN,NaN,NaN,2.728128,3.951082,1.222954,5.174036,"[TAX_FNCACT_SEAMAN,555, TAX_FNCACT_VENDOR,2745..."


In [40]:
# Calcul du nombre de thèmes pour chaque entrée et stockage dans une nouvelle colonne 'themes_count'.
gkg["themes_count"] = gkg["themes"].apply(len)

In [41]:
# Vérification si les thèmes incluent des mots liés à la crise ('CRISIS' ou 'TERROR') et stockage dans 'has_crisis_theme'.
gkg["has_crisis_theme"] = gkg["themes"].apply(
    lambda x: any("CRISIS" in t or "TERROR" in t for t in x)
)

In [42]:
# Affichage du DataFrame 'gkg' après l'ajout de la colonne 'has_crisis_theme'.
gkg

,DATE,DocumentIdentifier,V2Themes,V2Locations,V2Persons,V2Organizations,V2Counts,SharingImage,RelatedImages,tone,positive_score,negative_score,polarity,themes,themes_count,has_crisis_theme
0,1970-01-01 05:37:31.202161500,https://fr.allafrica.com/stories/202512020485....,"SLFID_NATURAL_RESOURCES,1013;WB_471_ECONOMIC_G...",1#Niger#NG#NG##16#8#NG#44;1#Niger#NG#NG##16#8#...,NaN,"Authority Of Basin Of Niger Them,49;Authority ...",NaN,NaN,NaN,5.944056,5.944056,0.000000,5.944056,"[SLFID_NATURAL_RESOURCES,1013, WB_471_ECONOMIC...",39,True
1,1970-01-01 05:37:31.202161500,https://fr.allafrica.com/stories/202512020413....,NaN,1#Seychelles#SE#SE##-4.583333#55.666667#SE#167...,NaN,"Federation Madagascar,1180",NaN,NaN,NaN,1.259446,3.274559,2.015113,5.289673,[nan],1,False
2,1970-01-01 05:37:30.702100000,https://malijet.com/actualite-sur-afrique/3034...,"NATURAL_DISASTER_DROWNING,1574;SLFID_ECONOMIC_...",1#Benin#BN#BN##9.5#2.25#BN#885;1#Benin#BN#BN##...,NaN,"A Committee,112",KILL#2#citizens Benin#1#Togo#TO#TO#8#1.166667#...,NaN,NaN,-3.250000,2.750000,6.000000,8.750000,"[NATURAL_DISASTER_DROWNING,1574, SLFID_ECONOMI...",47,True
3,1970-01-01 05:37:31.019013000,https://fightnews.com/2025-pfl-africa-semifina...,"TAX_FNCACT_REFEREE,2609;TAX_FNCACT_REFEREE,452...",1#Cameroon#CM#CM##6#12#CM#1901;1#Cameroon#CM#C...,"Africa Smartcage,965;Africa Justin Clarke,2800...","Fighters League,5388",PROTEST#2014##1#Cameroon#CM#CM#6#12#CM#2837;TA...,NaN,NaN,-0.473934,3.696682,4.170616,7.867299,"[TAX_FNCACT_REFEREE,2609, TAX_FNCACT_REFEREE,4...",64,True
4,1970-01-01 05:37:31.019001500,https://theeagleonline.com.ng/cultural-capital...,"EPU_ECONOMY,3599;EPU_ECONOMY,5460;EPU_ECONOMY_...","1#Nigerians#NI#NI##10#8#NI#9474;4#Lagos, Lagos...","Bilal Akkouche,9797;Ladi Kwali,10469;Aina Onab...","Coronation Group,97;Coronation Group,1535;Coro...",NaN,https://theeagleonline.com.ng/wp-content/uploa...,NaN,5.846299,6.694955,0.848656,7.543612,"[EPU_ECONOMY,3599, EPU_ECONOMY,5460, EPU_ECONO...",61,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48036,1970-01-01 05:37:30.612160000,https://www.miragenews.com/iaea-fao-launch-ato...,"POVERTY,2111;WB_695_POVERTY,2111;WB_427_WATER_...","4#Loumbila, Region Du Plateau-Central, Burkina...","Rafael Mariano Grossi,2614;Dongxin Feng,3807","International Atomic Energy Agency,108;Centre ...",NaN,https://cdn1.miragenews.com/wp-content/uploads...,NaN,1.610738,4.697987,3.087248,7.785235,"[POVERTY,2111, WB_695_POVERTY,2111, WB_427_WAT...",105,True
48037,1970-01-01 05:37:30.612160000,https://www.aa.com.tr/en/world/south-africa-re...,"SANCTIONS,68;SANCTIONS,921;SANCTIONS,1428;CRIS...",1#Israeli#IS#IS##31.5#34.75#IS#2049;1#Benin#BN...,"Antonio Guterres,1742;Benjamin Netanyahu,2083;...","African Foreign Ministry,332;International Cri...",NaN,https://cdnuploads.aa.com.tr/uploads/Contents/...,NaN,-10.000000,2.121212,12.121212,14.242424,"[SANCTIONS,68, SANCTIONS,921, SANCTIONS,1428, ...",69,True
48038,1970-01-01 05:37:30.612134500,https://www.salon.com/2025/06/12/the-age-of-gl...,"SCIENCE,2325;TAX_FNCACT_SCIENTIST,2325;TAX_FNC...",1#Moldova#MD#MD##47#29#MD#12766;1#Somalia#SO#S...,"David Ben-Gurion,6873;Mahmoud Khalil,13522;Pat...","Peel Commission,6786;United Nations,11347;Unit...","KILL#20000##4#Gaza, Israel (General), Israel#I...",https://mediaproxy.salon.com/width/1200/https:...,NaN,-4.111153,2.740769,6.851922,9.592691,"[SCIENCE,2325, TAX_FNCACT_SCIENTIST,2325, TAX_...",236,True
48039,1970-01-01 05:37:30.612140000,https://hunan.voc.com.cn/news/202506/29689492....,"TAX_FNCACT_SEAMAN,555;TAX_FNCACT_VENDOR,2745;T...","4#Xiashou, Guangdong, China#CH#CH30#13057#22.3...","Zebian King,6593;Hill Sea,336","Park Joe,4813;Benin Confucius Institute,1615;J...",NaN,NaN,NaN,2.728128,3.951082,1.222954,5.174036,"[TAX_FNCACT_SEAMAN,555, TAX_FNCACT_VENDOR,2745...",118,True


In [43]:
# Suppression de la colonne originale 'V2Themes' car la colonne 'themes' a été créée avec les informations traitées.
gkg = gkg.drop(columns=['V2Themes'])

In [44]:
# Filtrage du DataFrame 'gkg' pour ne conserver que les entrées mentionnant 'Benin' dans 'V2Locations'.
# Cette ligne est redondante si la précédente filtration est déjà suffisante, mais elle assure la persistance du filtre.
gkg = gkg[gkg["V2Locations"].str.contains("Benin", na=False)]

In [45]:
# Filtrage du DataFrame 'gkg' pour ne conserver que les entrées mentionnant 'BN' (code pays du Bénin) dans 'V2Locations'.
# Cette opération est plus spécifique et peut être combinée avec le filtrage par 'Benin' si nécessaire.
gkg = gkg[gkg["V2Locations"].str.contains("BN", na=False)]

In [46]:
# Affichage du DataFrame 'gkg' après les filtrages géographiques.
gkg

,DATE,DocumentIdentifier,V2Locations,V2Persons,V2Organizations,V2Counts,SharingImage,RelatedImages,tone,positive_score,negative_score,polarity,themes,themes_count,has_crisis_theme
0,1970-01-01 05:37:31.202161500,https://fr.allafrica.com/stories/202512020485....,1#Niger#NG#NG##16#8#NG#44;1#Niger#NG#NG##16#8#...,NaN,"Authority Of Basin Of Niger Them,49;Authority ...",NaN,NaN,NaN,5.944056,5.944056,0.000000,5.944056,"[SLFID_NATURAL_RESOURCES,1013, WB_471_ECONOMIC...",39,True
1,1970-01-01 05:37:31.202161500,https://fr.allafrica.com/stories/202512020413....,1#Seychelles#SE#SE##-4.583333#55.666667#SE#167...,NaN,"Federation Madagascar,1180",NaN,NaN,NaN,1.259446,3.274559,2.015113,5.289673,[nan],1,False
2,1970-01-01 05:37:30.702100000,https://malijet.com/actualite-sur-afrique/3034...,1#Benin#BN#BN##9.5#2.25#BN#885;1#Benin#BN#BN##...,NaN,"A Committee,112",KILL#2#citizens Benin#1#Togo#TO#TO#8#1.166667#...,NaN,NaN,-3.250000,2.750000,6.000000,8.750000,"[NATURAL_DISASTER_DROWNING,1574, SLFID_ECONOMI...",47,True
3,1970-01-01 05:37:31.019013000,https://fightnews.com/2025-pfl-africa-semifina...,1#Cameroon#CM#CM##6#12#CM#1901;1#Cameroon#CM#C...,"Africa Smartcage,965;Africa Justin Clarke,2800...","Fighters League,5388",PROTEST#2014##1#Cameroon#CM#CM#6#12#CM#2837;TA...,NaN,NaN,-0.473934,3.696682,4.170616,7.867299,"[TAX_FNCACT_REFEREE,2609, TAX_FNCACT_REFEREE,4...",64,True
4,1970-01-01 05:37:31.019001500,https://theeagleonline.com.ng/cultural-capital...,"1#Nigerians#NI#NI##10#8#NI#9474;4#Lagos, Lagos...","Bilal Akkouche,9797;Ladi Kwali,10469;Aina Onab...","Coronation Group,97;Coronation Group,1535;Coro...",NaN,https://theeagleonline.com.ng/wp-content/uploa...,NaN,5.846299,6.694955,0.848656,7.543612,"[EPU_ECONOMY,3599, EPU_ECONOMY,5460, EPU_ECONO...",61,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48036,1970-01-01 05:37:30.612160000,https://www.miragenews.com/iaea-fao-launch-ato...,"4#Loumbila, Region Du Plateau-Central, Burkina...","Rafael Mariano Grossi,2614;Dongxin Feng,3807","International Atomic Energy Agency,108;Centre ...",NaN,https://cdn1.miragenews.com/wp-content/uploads...,NaN,1.610738,4.697987,3.087248,7.785235,"[POVERTY,2111, WB_695_POVERTY,2111, WB_427_WAT...",105,True
48037,1970-01-01 05:37:30.612160000,https://www.aa.com.tr/en/world/south-africa-re...,1#Israeli#IS#IS##31.5#34.75#IS#2049;1#Benin#BN...,"Antonio Guterres,1742;Benjamin Netanyahu,2083;...","African Foreign Ministry,332;International Cri...",NaN,https://cdnuploads.aa.com.tr/uploads/Contents/...,NaN,-10.000000,2.121212,12.121212,14.242424,"[SANCTIONS,68, SANCTIONS,921, SANCTIONS,1428, ...",69,True
48038,1970-01-01 05:37:30.612134500,https://www.salon.com/2025/06/12/the-age-of-gl...,1#Moldova#MD#MD##47#29#MD#12766;1#Somalia#SO#S...,"David Ben-Gurion,6873;Mahmoud Khalil,13522;Pat...","Peel Commission,6786;United Nations,11347;Unit...","KILL#20000##4#Gaza, Israel (General), Israel#I...",https://mediaproxy.salon.com/width/1200/https:...,NaN,-4.111153,2.740769,6.851922,9.592691,"[SCIENCE,2325, TAX_FNCACT_SCIENTIST,2325, TAX_...",236,True
48039,1970-01-01 05:37:30.612140000,https://hunan.voc.com.cn/news/202506/29689492....,"4#Xiashou, Guangdong, China#CH#CH30#13057#22.3...","Zebian King,6593;Hill Sea,336","Park Joe,4813;Benin Confucius Institute,1615;J...",NaN,NaN,NaN,2.728128,3.951082,1.222954,5.174036,"[TAX_FNCACT_SEAMAN,555, TAX_FNCACT_VENDOR,2745...",118,True


In [47]:
# Affichage du DataFrame 'gkg' pour vérifier son état final après le nettoyage et les filtrages.
gkg

,DATE,DocumentIdentifier,V2Locations,V2Persons,V2Organizations,V2Counts,SharingImage,RelatedImages,tone,positive_score,negative_score,polarity,themes,themes_count,has_crisis_theme
0,1970-01-01 05:37:31.202161500,https://fr.allafrica.com/stories/202512020485....,1#Niger#NG#NG##16#8#NG#44;1#Niger#NG#NG##16#8#...,NaN,"Authority Of Basin Of Niger Them,49;Authority ...",NaN,NaN,NaN,5.944056,5.944056,0.000000,5.944056,"[SLFID_NATURAL_RESOURCES,1013, WB_471_ECONOMIC...",39,True
1,1970-01-01 05:37:31.202161500,https://fr.allafrica.com/stories/202512020413....,1#Seychelles#SE#SE##-4.583333#55.666667#SE#167...,NaN,"Federation Madagascar,1180",NaN,NaN,NaN,1.259446,3.274559,2.015113,5.289673,[nan],1,False
2,1970-01-01 05:37:30.702100000,https://malijet.com/actualite-sur-afrique/3034...,1#Benin#BN#BN##9.5#2.25#BN#885;1#Benin#BN#BN##...,NaN,"A Committee,112",KILL#2#citizens Benin#1#Togo#TO#TO#8#1.166667#...,NaN,NaN,-3.250000,2.750000,6.000000,8.750000,"[NATURAL_DISASTER_DROWNING,1574, SLFID_ECONOMI...",47,True
3,1970-01-01 05:37:31.019013000,https://fightnews.com/2025-pfl-africa-semifina...,1#Cameroon#CM#CM##6#12#CM#1901;1#Cameroon#CM#C...,"Africa Smartcage,965;Africa Justin Clarke,2800...","Fighters League,5388",PROTEST#2014##1#Cameroon#CM#CM#6#12#CM#2837;TA...,NaN,NaN,-0.473934,3.696682,4.170616,7.867299,"[TAX_FNCACT_REFEREE,2609, TAX_FNCACT_REFEREE,4...",64,True
4,1970-01-01 05:37:31.019001500,https://theeagleonline.com.ng/cultural-capital...,"1#Nigerians#NI#NI##10#8#NI#9474;4#Lagos, Lagos...","Bilal Akkouche,9797;Ladi Kwali,10469;Aina Onab...","Coronation Group,97;Coronation Group,1535;Coro...",NaN,https://theeagleonline.com.ng/wp-content/uploa...,NaN,5.846299,6.694955,0.848656,7.543612,"[EPU_ECONOMY,3599, EPU_ECONOMY,5460, EPU_ECONO...",61,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48036,1970-01-01 05:37:30.612160000,https://www.miragenews.com/iaea-fao-launch-ato...,"4#Loumbila, Region Du Plateau-Central, Burkina...","Rafael Mariano Grossi,2614;Dongxin Feng,3807","International Atomic Energy Agency,108;Centre ...",NaN,https://cdn1.miragenews.com/wp-content/uploads...,NaN,1.610738,4.697987,3.087248,7.785235,"[POVERTY,2111, WB_695_POVERTY,2111, WB_427_WAT...",105,True
48037,1970-01-01 05:37:30.612160000,https://www.aa.com.tr/en/world/south-africa-re...,1#Israeli#IS#IS##31.5#34.75#IS#2049;1#Benin#BN...,"Antonio Guterres,1742;Benjamin Netanyahu,2083;...","African Foreign Ministry,332;International Cri...",NaN,https://cdnuploads.aa.com.tr/uploads/Contents/...,NaN,-10.000000,2.121212,12.121212,14.242424,"[SANCTIONS,68, SANCTIONS,921, SANCTIONS,1428, ...",69,True
48038,1970-01-01 05:37:30.612134500,https://www.salon.com/2025/06/12/the-age-of-gl...,1#Moldova#MD#MD##47#29#MD#12766;1#Somalia#SO#S...,"David Ben-Gurion,6873;Mahmoud Khalil,13522;Pat...","Peel Commission,6786;United Nations,11347;Unit...","KILL#20000##4#Gaza, Israel (General), Israel#I...",https://mediaproxy.salon.com/width/1200/https:...,NaN,-4.111153,2.740769,6.851922,9.592691,"[SCIENCE,2325, TAX_FNCACT_SCIENTIST,2325, TAX_...",236,True
48039,1970-01-01 05:37:30.612140000,https://hunan.voc.com.cn/news/202506/29689492....,"4#Xiashou, Guangdong, China#CH#CH30#13057#22.3...","Zebian King,6593;Hill Sea,336","Park Joe,4813;Benin Confucius Institute,1615;J...",NaN,NaN,NaN,2.728128,3.951082,1.222954,5.174036,"[TAX_FNCACT_SEAMAN,555, TAX_FNCACT_VENDOR,2745...",118,True


In [48]:
# Calcul du nombre de valeurs manquantes dans la colonne 'SharingImage'.
gkg["SharingImage"].isna().sum()

np.int64(10427)

In [49]:
# Calcul du nombre de valeurs manquantes dans la colonne 'RelatedImages'.
gkg["RelatedImages"].isna().sum()

np.int64(44502)

La colonne `RelatedImages` contient un grand nombre de valeurs manquantes, ce qui la rend peu utile pour notre analyse. Nous allons la supprimer.

In [50]:
# Suppression de la colonne 'RelatedImages' en raison du nombre élevé de valeurs manquantes.
gkg = gkg.drop(columns=["RelatedImages"])

In [51]:
# Remplacement des valeurs manquantes dans 'SharingImage' par la chaîne 'no_image' pour indiquer l'absence d'image.
gkg['SharingImage'] = gkg['SharingImage'].fillna("no_image")

In [52]:
# Affichage du DataFrame 'gkg' après avoir traité les valeurs manquantes dans 'SharingImage'.
gkg

,DATE,DocumentIdentifier,V2Locations,V2Persons,V2Organizations,V2Counts,SharingImage,tone,positive_score,negative_score,polarity,themes,themes_count,has_crisis_theme
0,1970-01-01 05:37:31.202161500,https://fr.allafrica.com/stories/202512020485....,1#Niger#NG#NG##16#8#NG#44;1#Niger#NG#NG##16#8#...,NaN,"Authority Of Basin Of Niger Them,49;Authority ...",NaN,no_image,5.944056,5.944056,0.000000,5.944056,"[SLFID_NATURAL_RESOURCES,1013, WB_471_ECONOMIC...",39,True
1,1970-01-01 05:37:31.202161500,https://fr.allafrica.com/stories/202512020413....,1#Seychelles#SE#SE##-4.583333#55.666667#SE#167...,NaN,"Federation Madagascar,1180",NaN,no_image,1.259446,3.274559,2.015113,5.289673,[nan],1,False
2,1970-01-01 05:37:30.702100000,https://malijet.com/actualite-sur-afrique/3034...,1#Benin#BN#BN##9.5#2.25#BN#885;1#Benin#BN#BN##...,NaN,"A Committee,112",KILL#2#citizens Benin#1#Togo#TO#TO#8#1.166667#...,no_image,-3.250000,2.750000,6.000000,8.750000,"[NATURAL_DISASTER_DROWNING,1574, SLFID_ECONOMI...",47,True
3,1970-01-01 05:37:31.019013000,https://fightnews.com/2025-pfl-africa-semifina...,1#Cameroon#CM#CM##6#12#CM#1901;1#Cameroon#CM#C...,"Africa Smartcage,965;Africa Justin Clarke,2800...","Fighters League,5388",PROTEST#2014##1#Cameroon#CM#CM#6#12#CM#2837;TA...,no_image,-0.473934,3.696682,4.170616,7.867299,"[TAX_FNCACT_REFEREE,2609, TAX_FNCACT_REFEREE,4...",64,True
4,1970-01-01 05:37:31.019001500,https://theeagleonline.com.ng/cultural-capital...,"1#Nigerians#NI#NI##10#8#NI#9474;4#Lagos, Lagos...","Bilal Akkouche,9797;Ladi Kwali,10469;Aina Onab...","Coronation Group,97;Coronation Group,1535;Coro...",NaN,https://theeagleonline.com.ng/wp-content/uploa...,5.846299,6.694955,0.848656,7.543612,"[EPU_ECONOMY,3599, EPU_ECONOMY,5460, EPU_ECONO...",61,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48036,1970-01-01 05:37:30.612160000,https://www.miragenews.com/iaea-fao-launch-ato...,"4#Loumbila, Region Du Plateau-Central, Burkina...","Rafael Mariano Grossi,2614;Dongxin Feng,3807","International Atomic Energy Agency,108;Centre ...",NaN,https://cdn1.miragenews.com/wp-content/uploads...,1.610738,4.697987,3.087248,7.785235,"[POVERTY,2111, WB_695_POVERTY,2111, WB_427_WAT...",105,True
48037,1970-01-01 05:37:30.612160000,https://www.aa.com.tr/en/world/south-africa-re...,1#Israeli#IS#IS##31.5#34.75#IS#2049;1#Benin#BN...,"Antonio Guterres,1742;Benjamin Netanyahu,2083;...","African Foreign Ministry,332;International Cri...",NaN,https://cdnuploads.aa.com.tr/uploads/Contents/...,-10.000000,2.121212,12.121212,14.242424,"[SANCTIONS,68, SANCTIONS,921, SANCTIONS,1428, ...",69,True
48038,1970-01-01 05:37:30.612134500,https://www.salon.com/2025/06/12/the-age-of-gl...,1#Moldova#MD#MD##47#29#MD#12766;1#Somalia#SO#S...,"David Ben-Gurion,6873;Mahmoud Khalil,13522;Pat...","Peel Commission,6786;United Nations,11347;Unit...","KILL#20000##4#Gaza, Israel (General), Israel#I...",https://mediaproxy.salon.com/width/1200/https:...,-4.111153,2.740769,6.851922,9.592691,"[SCIENCE,2325, TAX_FNCACT_SCIENTIST,2325, TAX_...",236,True
48039,1970-01-01 05:37:30.612140000,https://hunan.voc.com.cn/news/202506/29689492....,"4#Xiashou, Guangdong, China#CH#CH30#13057#22.3...","Zebian King,6593;Hill Sea,336","Park Joe,4813;Benin Confucius Institute,1615;J...",NaN,no_image,2.728128,3.951082,1.222954,5.174036,"[TAX_FNCACT_SEAMAN,555, TAX_FNCACT_VENDOR,2745...",118,True


In [53]:
# Affichage des dimensions finales (nombre de lignes et de colonnes) des DataFrames 'events' et 'gkg' après le nettoyage.
print(events.shape)
print(gkg.shape)

(6353, 16)
(47775, 14)


In [54]:
# Sauvegarde des DataFrames nettoyés 'events' et 'gkg' en fichiers CSV dans Google Drive.
"""
events.to_csv('/content/drive/MyDrive/Datasets_Cleans_Isheero/events_clean.csv', index=False)
gkg.to_csv('/content/drive/MyDrive/Datasets_Cleans_Isheero/gkg_clean.csv', index=False)
""

SyntaxError: incomplete input (2729717547.py, line 2)